In [ ]:
# ===== BASELINE GREEDY — CONFIG (edit ONLY this cell) =====================

REPO_URL = "https://github.com/Avi161/ACSolverX.git"
BRANCH   = "test/stable-ac-moves-w4"   # branch that contains experiments/
REPO_DIR = "ACSolverX"
CLONE       = True
UPDATE_REPO = True           # git reset --hard so a RESTART pulls latest push
MOUNT_DRIVE = True           # results -> Drive if True, else local ./results/

# --- experiment knobs ---------------------------------------------------
BUDGET = [10000]   # node budget per presentation; one jsonl per budget

cfg = {
    "DATASET": "data/ms640_solved.txt",
    "SUBSET": None,                 # None=all 640, list[int], or (start, end)
    "MAX_RELATOR_LENGTH": 24,       # per-relator cap (24 = ms640 layout)
    "CYCLIC_REDUCE": True,          # toggle cyclic reduction after each move

    # jsonl field toggles
    "use_min_relator_length": True,
    "use_min_relator": True,
    "use_max_relator_length": True,
    "use_max_relator": True,
    "use_time": True,
    "use_path": True,
    "PATH_IN_SEPARATE_FILE": True,  # solved paths -> *_paths.jsonl

    "RESUME": True,                 # skip pres_ids already in the jsonl

    # output
    "MOUNT_DRIVE": MOUNT_DRIVE,
    "DRIVE_OUT_DIR": "/content/drive/acsolverx_results/greedy_baseline",
    "LOCAL_OUT_DIR": "results/greedy_baseline",

    # Weights & Biases
    "USE_WANDB": True,
    "WANDB_ENTITY": "avigyapaudel045",
    "WANDB_PROJECT": "acsolverx-greedy",
    "WANDB_MODE": "online",         # "offline" for flaky Colab -> wandb sync later
    "WANDB_GROUP": None,            # default: greedy_baseline_{date}
}

print("config loaded.")

In [ ]:
# ==================== SETUP (clone / pull / install / mount) ==============
import os, sys, subprocess

def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-2000:])
    if p.returncode != 0 and p.stderr: print("STDERR:", p.stderr[-2000:])

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab:", IN_COLAB)

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        if CLONE:
            sh(f"git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}")
    elif UPDATE_REPO:
        sh(f"cd {REPO_DIR} && git fetch --depth 1 origin {BRANCH} && git reset --hard FETCH_HEAD")
    sh(f"cd {REPO_DIR} && git log -1 --oneline")
    sh("pip -q install numba numpy wandb")
    if MOUNT_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
    REPO_ROOT = os.path.abspath(REPO_DIR)
else:
    # local: assume this notebook lives in ACSolverX/experiments/
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

# run from repo root so "data/..." and "import experiments..." resolve
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("repo root:", REPO_ROOT)

# W&B auth: prefer a Colab Secret named WANDB_API_KEY (add it via the key
# icon in Colab's left sidebar and toggle notebook access ON). If it's not
# found, fall back to the interactive wandb.login() prompt.
if cfg["USE_WANDB"]:
    import wandb
    if not os.environ.get("WANDB_API_KEY"):
        _key = None
        try:
            from google.colab import userdata
            _key = userdata.get("WANDB_API_KEY")
        except Exception:
            _key = None
        if _key:
            os.environ["WANDB_API_KEY"] = _key
            print("W&B: using Colab Secret WANDB_API_KEY (no prompt).")
        else:
            print("W&B: no WANDB_API_KEY secret found -> prompting.")
            wandb.login()

In [ ]:
# ==================== RUN =================================================
from experiments.run_baseline import run_dataset

for budget in BUDGET:
    run_dataset(cfg, node_budget=budget)   # resumable; one jsonl per budget